# Arabic Medical Question Classification — Baseline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)]()

**Task:** Classify Arabic medical questions into 8 specialties.

**Metric:** Macro F1 × 100 (0–100 scale)

**Baseline strategy:** TF-IDF → KNN with 5-fold stratified cross-validation → ensemble via majority vote.

In [ ]:
!pip install -q kagglehub

In [ ]:
# ── DO NOT MODIFY THIS CELL ──────────────────────────────────────────
import os
import kagglehub
import pandas as pd
import numpy as np

COMP_DATASET = "sattamjaltwaim/arabic-medical-dataset"  # UPDATE THIS SLUG

csv_root = kagglehub.dataset_download(COMP_DATASET)

train_csv = pd.read_csv(os.path.join(csv_root, "train.csv"))
test_csv  = pd.read_csv(os.path.join(csv_root, "test.csv"))
label_map = pd.read_csv(os.path.join(csv_root, "label_map.csv"))

idx_to_name = dict(zip(label_map["label_index"], label_map["label_name"]))
NUM_CLASSES = len(idx_to_name)

print(f"Train: {train_csv.shape}  |  Test: {test_csv.shape}  |  Classes: {NUM_CLASSES}")
print(f"\nLabel map:")
for idx, name in idx_to_name.items():
    print(f"  {idx}: {name}")

## Quick EDA

In [ ]:
import matplotlib.pyplot as plt

# Label distribution
counts = train_csv["label"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(counts.index, counts.values)
ax.set_xticks(counts.index)
ax.set_xticklabels([idx_to_name[i] for i in counts.index], rotation=45, ha="right")
ax.set_ylabel("Count")
ax.set_title("Training Label Distribution")
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 10, str(v),
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

# Question length stats
train_csv["q_len"] = train_csv["question"].str.len()
print(f"Question length — min: {train_csv['q_len'].min()}, "
      f"median: {train_csv['q_len'].median():.0f}, "
      f"max: {train_csv['q_len'].max()}")

# Show a few sample questions
print("\nSample questions:")
for _, row in train_csv.sample(3, random_state=42).iterrows():
    print(f"  [{idx_to_name[row['label']]}] {row['question'][:120]}...")

## Baseline: TF-IDF + KNN

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=10_000, sublinear_tf=True)

all_questions = pd.concat([train_csv["question"], test_csv["question"]], ignore_index=True)
vectorizer.fit(all_questions)

X_train = vectorizer.transform(train_csv["question"])
X_test  = vectorizer.transform(test_csv["question"])
y_train = train_csv["label"].values

print(f"TF-IDF shape — train: {X_train.shape}, test: {X_test.shape}")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(y_train), dtype=int)
test_preds_all = np.zeros((len(test_csv), N_SPLITS), dtype=int)
fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train[tr_idx], X_train[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    knn = KNeighborsClassifier(n_neighbors=5, metric="cosine")
    knn.fit(X_tr, y_tr)

    val_pred = knn.predict(X_val)
    oof_preds[val_idx] = val_pred

    fold_f1 = f1_score(y_val, val_pred, average="macro")
    fold_scores.append(fold_f1)
    print(f"  Fold {fold}: Macro F1 = {fold_f1:.4f}")

    test_preds_all[:, fold] = knn.predict(X_test)

overall_f1 = f1_score(y_train, oof_preds, average="macro")
print(f"\nOOF Macro F1:  {overall_f1:.4f}")
print(f"Mean fold F1:  {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
print(f"Score (×100):  {overall_f1 * 100:.2f}")

## Test Prediction (Majority Vote Ensemble)

In [ ]:
from collections import Counter

all_preds = []
for i in range(len(test_csv)):
    votes = test_preds_all[i, :]
    majority = Counter(votes).most_common(1)[0][0]
    all_preds.append(majority)

all_preds = np.array(all_preds)
print(f"Test predictions: {len(all_preds)} questions")
print(f"Predicted label distribution:")
for cls in range(NUM_CLASSES):
    n = (all_preds == cls).sum()
    print(f"  {cls} ({idx_to_name[cls]}): {n}")

## Generate Submission

In [ ]:
# ── DO NOT MODIFY THIS CELL ──────────────────────────────────────────
def generate_submission(predictions, filename="submission.csv"):
    """Create a Kaggle submission file from predictions."""
    y = np.asarray(predictions, dtype=int)
    assert len(y) == len(test_csv), (
        f"Expected {len(test_csv)} predictions, got {len(y)}"
    )
    submission = pd.DataFrame({
        "id": test_csv["id"].values,
        "prediction": y.astype(float),
    })
    submission.to_csv(filename, index=False)
    print(f"Saved {filename}  ({len(submission)} rows)")
    print(submission.head())

generate_submission(all_preds)